<a href="https://colab.research.google.com/github/nehakeer/Automation-of-Attendance-System-to-Recognize-a-Human-Frontal-Face-With-and-Without-Masks/blob/main/RAG/LLM_Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://openrouter.ai/settings/keys

In [13]:
!pip install langchain_community -q

In [14]:
!pip install langchain-experimental -q

In [15]:
!pip install pdfplumber -q

In [16]:
!pip install faiss-cpu -q

In [17]:
!git clone https://github.com/deepanrajm/deep_learning.git

fatal: destination path 'deep_learning' already exists and is not an empty directory.


In [18]:
!pip install langchain-openai -q

In [20]:
import os
from langchain_openai import ChatOpenAI # Updated import
from langchain_core.messages import HumanMessage

# Step 1: Set your OpenRouter API key and endpoint
os.environ["OPENAI_API_KEY"] = "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"  # Replace with your actual OpenRouter API key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Step 2: Initialize the LLM
llm = ChatOpenAI(
    model= "nvidia/nemotron-3-nano-30b-a3b:free", #"upstage/solar-pro-3:free",  # or try "meta-llama/llama-3-8b-instruct", etc.
    temperature=0.7, # 0 creativity 1 most creativity 0-1 range
    openai_api_base=os.environ["OPENAI_API_BASE"],
    openai_api_key=os.environ["OPENAI_API_KEY"],
    request_timeout=60, #server becomes unresponsive after 60 secs
    verbose=True
)

# Step 3: Interact with the model (normal chat mode)
while True:
    query = input("\nYou: ")
    if query.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    response = llm.invoke([HumanMessage(content=query)]) # Corrected method call
    print("\nAssistant:", response.content) #only the output from json file response.content will show


You: remedy for cough?

Assistant: **Quick‑Start Guide to Soothing a Cough**

| Situation | What to Try | How to Use | Why It Helps |
|-----------|------------|------------|--------------|
| **Mild, acute cough (no fever, shortness of breath, chest pain)** | • Warm honey‑lemon tea  <br>• Steam inhalation  <br>• Saline nasal spray  <br>• Over‑the‑counter (OTC) expectorant (guaifenesin) or suppressant (dextromethorphan) | • Mix 1–2 tsp honey with warm water/tea (don’t give honey to kids < 1 yr). <br>• Inhale steam from a bowl of hot water or a humidifier for 5–10 min, covering head with a towel. <br>• Use a nasal spray 2–3× daily to clear mucus. <br>• Follow package dosing (usually 100–200 mg guaifenesin every 4 h, or 10–20 mg dextromethorphan every 4 h). | Honey coats the throat and has mild antimicrobial/anti‑inflammatory properties. Steam moistens irritated airways and loosens mucus. Expectorants thin mucus so it’s easier to clear; suppressants reduce the cough reflex when it’s non‑p

KeyboardInterrupt: Interrupted by user

In [19]:
!pip install langchain-classic

In [21]:
import os
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS #FAISS is vector db

# Set your OpenRouter API key and endpoint
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Load and process PDF
loader = PDFPlumberLoader("/content/deep_learning/RAG/Basic_Home_Remedies.pdf")
docs = loader.load()  # page wise info
print("Pages loaded:", len(docs))

# Chunking
text_splitter = SemanticChunker(HuggingFaceEmbeddings()) # huggingface founder of LLMs - every LLM is available single hub for LLMs
documents = text_splitter.split_documents(docs)

# Vector DB
embedder = HuggingFaceEmbeddings()
vector = FAISS.from_documents(documents, embedder) # converting every semantic chunks into embedding and put into vector db
retriever = vector.as_retriever(search_type="similarity", search_kwargs={"k": 2}) # looks for closest embedding and retriever set to 2

# Use ChatOpenAI with OpenRouter backend
llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",   #"mistralai/mistral-7b-instruct"
    temperature=0.7,
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    request_timeout=60,
)

# Prompt
prompt = """
You are a helpful assistant.
Use the following pieces of context to answer the question at the end.
Answer only using the context and be concise (3–4 sentences).

Context: {context}

Question: {question}

Answer:
"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt)

llm_chain = LLMChain(llm=llm, prompt=QA_CHAIN_PROMPT, verbose=True)

document_prompt = PromptTemplate(
    input_variables=["page_content", "source"],
    template="Context:\ncontent:{page_content}\nsource:{source}",
)

combine_documents_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="context",
    document_prompt=document_prompt,
    callbacks=None,
)

qa = RetrievalQA(
    combine_documents_chain=combine_documents_chain,
    retriever=retriever,
    return_source_documents=True,
    verbose=True,
)

# Example query
result = qa("remedy for cough?")
print("Answer:", result["result"])


Pages loaded: 3


/tmp/ipykernel_210/2342532132.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  text_splitter = SemanticChunker(HuggingFaceEmbeddings()) # huggingface founder of LLMs - every LLM is available single hub for LLMs
/tmp/ipykernel_210/2342532132.py:22: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  text_splitter = SemanticChunker(HuggingFaceEmbeddings()) # huggingface founder of LLMs - every LLM is available single hub for LLMs
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_210/2342532132.py:26: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedder = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_210/2342532132.py:31: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
/tmp/ipykernel_210/2342532132.py:53: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=QA_CHAIN_PROMPT, verbose=True)
/tmp/ipykernel_210/23425321



> Entering new RetrievalQA chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are a helpful assistant.
Use the following pieces of context to answer the question at the end.
Answer only using the context and be concise (3–4 sentences).

Context: Context:
content:Basic Home Remedies & Healthy Lifestyle
Guide
1. Introduction
Home remedies have been used for centuries to treat common ailments using natural ingredients. Combined with a healthy lifestyle, these remedies can support overall well-being and disease
prevention. 2. Home Remedies for Common Ailments
2.1 Cold & Flu
 Ginger Tea: Boil ginger slices in water, add honey and lemon. Helps relieve congestion
and sore throat.  Turmeric Milk: Mix 1 tsp turmeric in warm milk. Acts as an anti-inflammatory and
boosts immunity.  Steam Inhalation: Boil water, add eucalyptus oil, and inhale steam to clear nasal
congestion. 2.2 Cough
 Honey & Lemon: Mix 1 tbsp honey with warm water and lemon juice. Soothes the
throat 

In [ ]:
import os
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS #FAISS is vector db

# Set your OpenRouter API key and endpoint
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Load and process PDF
loader = PDFPlumberLoader("/content/deep_learning/RAG/Basic_Home_Remedies.pdf")
docs = loader.load()  # page wise info
print("Pages loaded:", len(docs))

# Chunking
text_splitter = SemanticChunker(HuggingFaceEmbeddings()) # huggingface founder of LLMs - every LLM is available single hub for LLMs
documents = text_splitter.split_documents(docs)

# Vector DB
embedder = HuggingFaceEmbeddings()
vector = FAISS.from_documents(documents, embedder) # converting every semantic chunks into embedding and put into vector db
retriever = vector.as_retriever(search_type="similarity", search_kwargs={"k": 2}) # looks for closest embedding and retriever set to 2

# Use ChatOpenAI with OpenRouter backend
llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",   #"mistralai/mistral-7b-instruct"
    temperature=0.7,
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    request_timeout=60,
)

# Prompt
prompt = """
You are an experience doctor in home remedies
Use the following pieces of context to answer the question at the end.
Answer only using the context and be concise (3–4 sentences).
Answer always like a doctor and prescribe medicines


Context: {context}

Question: {question}

Answer:
"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt)

llm_chain = LLMChain(llm=llm, prompt=QA_CHAIN_PROMPT, verbose=True)

document_prompt = PromptTemplate(
    input_variables=["page_content", "source"],
    template="Context:\ncontent:{page_content}\nsource:{source}",
)

combine_documents_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="context",
    document_prompt=document_prompt,
    callbacks=None,
)

qa = RetrievalQA(
    combine_documents_chain=combine_documents_chain,
    retriever=retriever,
    return_source_documents=True,
    verbose=True,
)

# Example query
result = qa("remedy for cough?")
print("Answer:", result["result"])
